# Regresión con Embedding
##** Estimación de Biomasa Aérea (AGB) - AOI: Manglares, Bahía Málaga **



## 0. Instalación y autenticación

In [ ]:
!pip install earthengine-api geemap -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 47.4 MB/s eta 0:00:00


In [ ]:
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Autenticar e inicializar Earth Engine.
# Reemplaza 'tu-proyecto-gee' por el ID de tu proyecto de Google Cloud
# asociado a Earth Engine (Cloud Project).
ee.Authenticate()
ee.Initialize(project='ee-paulapaz1101')

EEException: Caller does not have required permission to use project ee-paulapaz1101. Grant the caller the roles/serviceusage.serviceUsageConsumer role, or a custom role with the serviceusage.services.use permission, by visiting https://console.developers.google.com/iam-admin/iam?project=ee-paulapaz1101 and then retry. Propagation of the new permission may take a few minutes.

In [ ]:
# Crear el mapa interactivo (equivalente al Map del Code Editor)
Map = geemap.Map()
Map.add_basemap('SATELLITE')
Map

## 1. Dibujar el polígono de la zona de interés (ROI)

Usa las herramientas de dibujo del mapa de arriba (ícono de polígono, a la izquierda)
para dibujar tu propia zona de interés directamente sobre el mapa satelital.

Pasos:
1. Sube o desplázate en el mapa de la celda anterior hasta tu zona de interés.
2. Haz clic en el ícono de polígono (o rectángulo) en la barra de herramientas del mapa.
3. Dibuja el contorno de tu zona (por ejemplo, la bahía y las zonas de manglar).
4. Ejecuta la celda de abajo: tomará automáticamente lo que dibujaste (`Map.user_roi`).

In [ ]:
# # Polígono de respaldo (Bahía Málaga) por si no se dibuja nada en el mapa
# geometry_default = ee.Geometry.Polygon([
#     [
#         [-77.45, 4.15],
#         [-77.30, 4.15],
#         [-77.30, 3.90],
#         [-77.45, 3.90],
#         [-77.45, 4.15]
#     ]
# ])

# Si dibujaste un poligono en el mapa (Map.user_roi), se usa ese;
# si no, se usa el poligono de respaldo de Bahia Malaga.
if Map.user_roi is not None:
    geometry = Map.user_roi
    print('Usando el poligono dibujado en el mapa.')
else:
    geometry = geometry_default
    print('No se detecto ningun poligono dibujado; usando el poligono por defecto (Bahia Malaga).')

Map.centerObject(geometry, 11)

> **Nota:** si quieres cargar tu ROI desde un archivo (por ejemplo un `.geojson`, `.shp` o
> `.kml`) en vez de dibujarlo a mano, puedes usar `geemap.geojson_to_ee('ruta.geojson')`
> o `geemap.shp_to_ee('ruta.shp')` y asignar el resultado directamente a la variable
> `geometry`.

## 2. Definir periodo de tiempo

In [ ]:
startDate = ee.Date.fromYMD(2022, 1, 1)
endDate = startDate.advance(1, 'year')

## Datos de entrada — dos fuentes satelitales:

- Satellite Embedding (GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL): 64 bandas que
resumen información espectral/espacial anual de cada píxel — son las variables predictoras.
- GEDI L4A (LARSE/GEDI/GEDI04_A_002_MONTHLY): mediciones lidar de biomasa real (agbd, en Mg/ha) tomadas por el satélite ICESat-2/GEDI — es la variable a predecir. Se filtra por calidad (descarta pulsos degradados, alto error relativo y pendientes >30%).
## 3. Importar dataset de Satellite Embedding

In [ ]:
embeddings = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')

embeddingsFiltered = (
    embeddings
    .filter(ee.Filter.date(startDate, endDate))
    .filter(ee.Filter.bounds(geometry))
)

# Extraer la proyección de la primera banda de la primera imagen
embeddingsProjection = ee.Image(embeddingsFiltered.first()).select(0).projection()

# Establecer la proyección del mosaico
embeddingsImage = embeddingsFiltered.mosaic().setDefaultProjection(embeddingsProjection)

Map.addLayer(embeddingsImage, {}, 'embeddingsImage', False)

## 4. Importar dataset de mosaico GEDI L4A

In [ ]:
gedi = ee.ImageCollection('LARSE/GEDI/GEDI04_A_002_MONTHLY')

In [ ]:
# Función para seleccionar datos GEDI de alta calidad
# con el fin de descartar datos de baja calidad
# (nubes, pendientes pronunciadas, alto error relativo)
# donde: l4_quality_flag y degrade_flag: indicadores de calidad del pulso láser.
def qualityMask(image):
    return (
        image.updateMask(image.select('l4_quality_flag').eq(1))
        .updateMask(image.select('degrade_flag').eq(0))
    )


# Función para enmascarar mediciones GEDI poco confiables
# error estándar relativo > 50% => agbd_se / agbd > 0.5
# agbd: biomasa aérea estimada (Mg/ha) -> el valor que vamos a predecir
# donde agbd_se: error estándar de esa estimación.
def errorMask(image):
    relative_se = image.select('agbd_se').divide(image.select('agbd'))
    return image.updateMask(relative_se.lte(0.5))


# Función para enmascarar mediciones GEDI en pendientes > 30%
def slopeMask(image):
    glo30 = ee.ImageCollection('COPERNICUS/DEM/GLO30')

    glo30Filtered = (
        glo30
        .filter(ee.Filter.bounds(geometry))
        .select('DEM')
    )

    demProj = glo30Filtered.first().select(0).projection()

    elevation = glo30Filtered.mosaic().rename('dem').setDefaultProjection(demProj)

    slope = ee.Terrain.slope(elevation)

    return image.updateMask(slope.lt(30))

In [ ]:
gediFiltered = (
    gedi
    .filter(ee.Filter.date(startDate, endDate))
    .filter(ee.Filter.bounds(geometry))
)

gediProjection = ee.Image(gediFiltered.first()).select('agbd').projection()

gediProcessed = (
    gediFiltered
    .map(qualityMask)
    .map(errorMask)
    .map(slopeMask)
)

gediMosaic = gediProcessed.mosaic().select('agbd').setDefaultProjection(gediProjection)

# Visualizar el mosaico GEDI
gediVis = {
    'min': 0,
    'max': 200,
    'palette': ['#edf8fb', '#b2e2e2', '#66c2a4', '#2ca25f', '#006d2c'],
    'bands': ['agbd']
}

# Verifica en el mapa la disponibilidad de datos GEDI para el AOI
Map.addLayer(gediMosaic, gediVis, 'GEDI L4A (Filtrado)', True)
Map

## 5. Remuestrear imagen embedding y GEDI, y poner en grilla de 100m
Ambas fuentes tienen resoluciones distintas, así que se remuestrean juntas a una grilla común de 100 m (EPSG:3857), promediando con reduceResolution, y se exportan como un asset intermedio.

In [ ]:
# Definir la grilla de destino: proyección EPSG:3857 con píxeles de 100 metros
gridScale = 100
gridProjection = ee.Projection('EPSG:3857').atScale(gridScale)

# Juntar las dos imágenes en una sola: las 64 bandas del embedding + la banda
# agbd de GEDI, todo en el mismo objeto. Así se remuestrean juntas y quedan
# perfectamente alineadas. Usar el método de remuestreo 'bilinear'.
stacked = embeddingsImage.addBands(gediMosaic)
stacked = stacked.resample('bilinear')

stackedResampled = (
    stacked
    .reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=1024  # límite de píxeles a promediar por píxel grande
                         # (100 m / 10 m = 10x10 = 100 píxeles, 1024 da margen).
    )
    .reproject(crs=gridProjection)
)

stackedResampled = stackedResampled.updateMask(stackedResampled.mask().gt(0))

### Exportar el mosaico combinado a un asset

**IMPORTANTE**: reemplaza `exportFolder` con tu propia carpeta de assets de GEE.
La carpeta debe existir antes de exportar (ej. `projects/tu-proyecto-gee/assets/biodiversidad-taller/`).

In [ ]:
exportFolder = 'projects/ee-paulapaz1101/assets/biodiversity_workshop/GEODATA/satellite_embedding/'
mosaicExportImage = 'gedi_mosaic_bahia_malaga3'
mosaicExportImagePath = exportFolder + mosaicExportImage

task_mosaic = ee.batch.Export.image.toAsset(
    image=stackedResampled.clip(geometry),
    description='GEDI_Mosaic_Bahia_Malaga_Export',
    assetId=mosaicExportImagePath,
    region=geometry,
    scale=gridScale,
    maxPixels=1e10
)
task_mosaic.start()
print('Tarea de exportación iniciada:', task_mosaic.id)

**Espera a que la tarea termine** antes de seguir. Ejecuta la siguiente celda para monitorear
el estado (puedes volver a correrla cada tanto hasta que diga `COMPLETED`).

In [ ]:
import time

while True:
    status = task_mosaic.status()
    state = status['state']
    print('Estado:', state)
    if state in ('COMPLETED', 'FAILED', 'CANCELLED'):
        print(status)
        break
    time.sleep(30)

In [ ]:
# Una vez que la exportación haya terminado, continúa con el mosaico
# exportado desde aquí en adelante.
stackedResampled = ee.Image(mosaicExportImagePath)

## 6. Extraer características de entrenamiento
Se extraen ~1000 puntos donde sí hay datos GEDI válidos (stratifiedSample), generando una tabla con las 64 bandas de embedding + el valor real de agbd en cada punto.

In [ ]:
# Define cuáles son las variables de entrada (predictors - las 64 bandas
# del Satellite Embedding) y la variable a predecir (predicted - el nombre
# de la única banda de GEDI, que es 'agbd').
predictors = embeddingsImage.bandNames()
predicted = gediMosaic.bandNames().get(0)
print('predictors', predictors.getInfo())
print('predicted', predicted.getInfo())

predictorImage = stackedResampled.select(predictors)
predictedImage = stackedResampled.select([predicted])

classMask = predictedImage.mask().toInt().rename('class')

In [ ]:
numSamples = 1000

# Agrega la banda 'class' a la imagen completa.
# stratifiedSample toma muestras por clase, no al azar en toda la imagen.
# classValues: [0, 1] -> existen dos clases posibles (0 = sin dato, 1 = con dato).
# classPoints: [0, numSamples] -> 0 puntos de la clase 0 (vacíos) y 1000 puntos
# de la clase 1 (donde sí hay GEDI), asegurando que las muestras caigan
# exactamente donde hay información real.
# dropNulls: True -> descarta cualquier muestra con algún valor nulo en alguna banda.
# tileScale: 16 -> parámetro técnico de rendimiento, divide el cálculo en tareas
# más pequeñas para evitar que el servidor de GEE se quede sin memoria.

training = stackedResampled.addBands(classMask).stratifiedSample(
    numPoints=numSamples,
    classBand='class',
    region=geometry,
    scale=gridScale,
    classValues=[0, 1],
    classPoints=[0, numSamples],
    dropNulls=True,
    tileScale=16,
)

print('Número de features extraídas', training.size().getInfo())
print('Feature de entrenamiento (ejemplo)', training.first().getInfo())

In [ ]:
# Estadísticas descriptivas de agbd en los puntos de entrenamiento (GEDI)
agbd_stats = training.aggregate_stats('agbd')
print('Estadísticas de agbd (Mg/ha):', agbd_stats.getInfo())

## 7. Entrenar modelo de regresión
Se entrena un Random Forest de regresión (smileRandomForest, 50 árboles) que aprende a predecir agbd a partir de las 64 bandas de embedding. Se valida con RMSE y un gráfico observado vs. predicho (con línea de tendencia y R²).

In [ ]:
model = ee.Classifier.smileRandomForest(50).setOutputMode('REGRESSION').train(
    features=training,
    classProperty=predicted,
    inputProperties=predictors
)

predictedFc = training.classify(
    classifier=model,
    outputName='agbd_predicted'
)

In [ ]:
# Calcular RMSE
def calculateRmse(input_fc):
    observed = ee.Array(input_fc.aggregate_array('agbd'))
    predicted_arr = ee.Array(input_fc.aggregate_array('agbd_predicted'))
    rmse = observed.subtract(predicted_arr).pow(2).reduce('mean', [0]).sqrt().get([0])
    return rmse


rmse = calculateRmse(predictedFc)
print('RMSE', rmse.getInfo())

Un RMSE de 51.22 Mg/ha con estos datos es un resultado razonablemente bueno, no excelente. El modelo captura una parte sustancial de la variabilidad de biomasa, pero todavía deja un margen de error considerable. Teniendo en cuenta el valor de la media de agbd de datos de entrenamiento 155.5%, el RMSE relativo a la media = 51.22 / 155.5 ≈ 33%, es un error que equivale a un tercio del valor promedio de biomasa. Si revisamos el RMSE relativo al rango total= 51.22 / 831.1 ≈ 6.2%, sobre el rango completo de valores observados, el error es relativamente pequeño.

Si queremos mejorar el modelo, podemos:

1. Aumentar el número de muestras de entrenamiento (probar con 2000–3000).
2. Ajustar hiperparámetros del Random Forest (más árboles, variablesPerSplit, minLeafPopulation).
3. Revisar si hay valores extremos en datos agbd (cerca de 831 Mg/ha) que estén afectando desproporcionadamente el error.

### Gráfico de valores observados vs. predichos

Equivalente al `ui.Chart.feature.byFeature` del Code Editor, usando `matplotlib`.

In [ ]:
chart_data = predictedFc.select(['agbd', 'agbd_predicted']).getInfo()
rows = [f['properties'] for f in chart_data['features']]
df = pd.DataFrame(rows)

x = df['agbd'].values
y = df['agbd_predicted'].values

# Ajuste lineal (trendline) y R^2
coeffs = np.polyfit(x, y, 1)
trend = np.poly1d(coeffs)
residuals = y - trend(x)
ss_res = np.sum(residuals**2)
ss_tot = np.sum((y - np.mean(y))**2)
r2 = 1 - ss_res / ss_tot

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(x, y, marker='^', color='#525252', alpha=0.8, s=20)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, trend(x_line), color='black', linewidth=1,
         label=f'Ajuste Lineal (R² = {r2:.3f})')
ax.set_title('Densidad de Biomasa Aérea - Bahía Málaga (Mg/Ha)')
ax.set_xlabel('Observado')
ax.set_ylabel('Predicho')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

El ajuste líneal R2= 0.819 nos presenta la proporción de la variabilidad e los valores observados de agbd dada por el modelo. Este R2 significa que el modelo explica ~82% de la variabilidad en la biomasa observada por GEDI. El 18% restante es ruido error de medición de GEDI, o variabilidad que los datos Embedding no logra capturar.

## 8. Generar predicciones
El modelo entrenado se aplica a toda la imagen (no solo los puntos de muestra), generando un mapa continuo de biomasa predicha para toda el área.

In [ ]:
predictedImage = stackedResampled.classify(
    classifier=model,
    outputName='agbd'
)

**IMPORTANTE:** Estas métricas de evaluación del modelo R² (y el RMSE) se calcularon sobre los mismos 1000 puntos que usaste para entrenar el modelo (training). Eso significa que el número probablemente esté optimista/sobreestimado, porque el Random Forest ya "aprendió" esos datos durante el entrenamiento, no es una validación independiente.

Para saber si el 0.819 es real o inflado por sobreajuste, lo ideal es:

1. Dividir tus puntos en entrenamiento (70-80%) y validación (20-30%) antes de entrenar.
2. Entrenar solo con el conjunto de entrenamiento.
3. Calcular RMSE y R² solo sobre el conjunto de validación (nuevos datos para el modelo).

## 9. Exportar la imagen con valores predichos

**IMPORTANTE**: usa la misma `exportFolder` definida arriba (o cambia la ruta según tu proyecto).

In [ ]:
predictedExportImage = 'predicted_agbd_bahia_malaga3'
predictedExportImagePath = exportFolder + predictedExportImage

task_predicted = ee.batch.Export.image.toAsset(
    image=predictedImage.clip(geometry),
    description='Predicted_AGBD_Bahia_Malaga_Export',
    assetId=predictedExportImagePath,
    region=geometry,
    scale=gridScale,
    maxPixels=1e10
)
task_predicted.start()
print('Tarea de exportación iniciada:', task_predicted.id)

**Esperar a que la tarea termine** antes de seguir (vuelve a correr la celda si sigue en `RUNNING`).

In [ ]:
while True:
    status = task_predicted.status()
    state = status['state']
    print('Estado:', state)
    if state in ('COMPLETED', 'FAILED', 'CANCELLED'):
        print(status)
        break
    time.sleep(30)

In [ ]:
# Espera a que termine de exportar antes de continuar.
# Luego visualizar la imagen exportada.
predictedImage = ee.Image(predictedExportImagePath)

gediVis = {
    'min': 0,
    'max': 200,
    'palette': ['#edf8fb', '#b2e2e2', '#66c2a4', '#2ca25f', '#006d2c'],
    'bands': ['agbd']
}

Map.addLayer(predictedImage, gediVis, 'AGBD Predicho - Bahía Málaga')
Map

## 10. Estimar la biomasa total
Se usa ESA WorldCover para quedarse solo con píxeles de vegetación relevante (bosques, arbustos, pastizales, cultivos, manglares) y descartar agua, suelo desnudo, zonas urbanas, etc.

In [ ]:
# Usamos ESA WorldCover v200 para seleccionar coberturas de vegetación
# (incluye la clase 95: Manglares)
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first()

worldcoverResampled = (
    worldcover
    .reduceResolution(reducer=ee.Reducer.mode(), maxPixels=1024)
    .reproject(crs=gridProjection)
)

| Clase       | Valor |
|-------------|-------|
| Bosques     | 10    |
| Arbustos    | 20    |
| Pastizales  | 30    |
| Cultivos    | 40    |
| Manglares   | 95    |

In [ ]:
landCoverMask = (
    worldcoverResampled.eq(10)
    .Or(worldcoverResampled.eq(20))
    .Or(worldcoverResampled.eq(30))
    .Or(worldcoverResampled.eq(40))
    .Or(worldcoverResampled.eq(95))
)


predictedImageMasked = predictedImage.updateMask(landCoverMask)
Map.addLayer(predictedImageMasked, gediVis, 'AGBD Predicho (Enmascarado)')
Map

## 11. Calcular la Biomasa Total
Se multiplica la densidad predicha (Mg/ha) por el área de cada píxel para obtener toneladas totales, y se suman todos los píxeles enmascarados dentro del ROI biomasa total estimada (Mg) de la zona.

In [ ]:
pixelAreaHa = ee.Image.pixelArea().divide(10000)
predictedAgb = predictedImageMasked.multiply(pixelAreaHa)

stats = predictedAgb.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=geometry,
    scale=gridScale,
    maxPixels=1e10,
    tileScale=16
)

totalAgb = stats.getNumber('agbd')

print('Biomasa Total (Mg) - Bahía Málaga', totalAgb.getInfo())